# IPL Data Analysis - Data Cleaning

In [1]:
# Objective:
# Prepare and clean raw IPL datasets to ensure consistency and usability
# for further analysis. This includes handling missing values, fixing data types,
# and standardizing column names.

## Load datasets

In [ ]:
from google.colab import drive
import pandas as pd
import os

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Define the path to your file (Check if your folder name matches exactly)
file_path = '/content/drive/MyDrive/IPL_PROJECT/01_data_raw/IPL.csv'

# 3. Load the data
df = pd.read_csv(file_path)

# 4. Look at the first 5 rows to confirm it worked
print("Data Loaded Successfully!")
df.head()

Mounted at /content/drive


/tmp/ipykernel_1410/2096705788.py:12: DtypeWarning: Columns (28,29,30,31,43,46,47,48,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Data Loaded Successfully!


,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_runs,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,1,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,2,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


# Inspect dataset structure and columns

In [ ]:
# This will list all 64 columns clearly
print(f"Total Columns: {len(df.columns)}")
print("-" * 30)
for col in df.columns:
    print(col)

Total Columns: 64
------------------------------
Unnamed: 0
match_id
date
match_type
event_name
innings
batting_team
bowling_team
over
ball
ball_no
batter
bat_pos
runs_batter
balls_faced
bowler
valid_ball
runs_extras
runs_total
runs_bowler
runs_not_boundary
extra_type
non_striker
non_striker_pos
wicket_kind
player_out
fielders
runs_target
review_batter
team_reviewed
review_decision
umpire
umpires_call
player_of_match
match_won_by
win_outcome
toss_winner
toss_decision
venue
city
day
month
year
season
gender
team_type
superover_winner
result_type
method
balls_per_over
overs
event_match_no
stage
match_number
team_runs
team_balls
team_wicket
new_batter
batter_runs
batter_balls
bowler_wicket
batting_partners
next_batter
striker_out


In [ ]:
# 1. Install DuckDB
!pip install duckdb -q

import duckdb
import pandas as pd

## Creating the working dataset with only the 'Essential ' columns

In [ ]:

df_working = duckdb.query("""
SELECT
    match_id,
    season,
    date,
    innings,
    batting_team,
    bowling_team,
    over,
    ball,
    batter,
    bowler,
    runs_batter,
    runs_bowler,
    runs_extras,
    runs_total,
    team_runs,
    wicket_kind,
    player_out,
    venue,
    city,
    toss_winner,
    toss_decision,
    player_of_match,
    match_won_by as winning_team,
    win_outcome as win_margin
FROM df
""").df()

print(f"Original Columns: {len(df.columns)} | Selected Columns: {len(df_working.columns)}")
df_working.head()

Original Columns: 64 | Selected Columns: 24


,match_id,season,date,innings,batting_team,bowling_team,over,ball,batter,bowler,...,team_runs,wicket_kind,player_out,venue,city,toss_winner,toss_decision,player_of_match,winning_team,win_margin
0,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,...,1,None,None,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs
1,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,...,1,None,None,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs
2,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,...,2,None,None,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs
3,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,...,2,None,None,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs
4,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,...,2,None,None,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs


# Standardize column names for consistency across analysis

## Standardizing the full list of IPL teams

In [ ]:

df_final = duckdb.query("""
SELECT *,
    CASE
        WHEN batting_team IN ('Kings XI Punjab', 'Punjab Kings') THEN 'PBKS'
        WHEN batting_team IN ('Delhi Daredevils', 'Delhi Capitals') THEN 'DC'
        WHEN batting_team IN ('Deccan Chargers', 'Sunrisers Hyderabad') THEN 'SRH'
        WHEN batting_team IN ('Rising Pune Supergiants', 'Rising Pune Supergiant') THEN 'RPS'
        WHEN batting_team = 'Gujarat Lions' THEN 'GL'
        WHEN batting_team IN ('Royal Challengers Bangalore','Royal Challengers Bengaluru' )THEN 'RCB'
        WHEN batting_team = 'Mumbai Indians' THEN 'MI'
        WHEN batting_team = 'Chennai Super Kings' THEN 'CSK'
        WHEN batting_team = 'Kolkata Knight Riders' THEN 'KKR'
        WHEN batting_team = 'Rajasthan Royals' THEN 'RR'
        WHEN batting_team = 'Lucknow Super Giants' THEN 'LSG'
        WHEN batting_team = 'Gujarat Titans' THEN 'GT'
        WHEN batting_team = 'Pune Warriors' THEN 'PWI'
        WHEN batting_team = 'Kochi Tuskers Kerala' THEN 'KTK'
        ELSE batting_team
    END AS bat_team_clean,
    CASE
        WHEN bowling_team IN ('Kings XI Punjab', 'Punjab Kings') THEN 'PBKS'
        WHEN bowling_team IN ('Delhi Daredevils', 'Delhi Capitals') THEN 'DC'
        WHEN bowling_team IN ('Deccan Chargers', 'Sunrisers Hyderabad') THEN 'SRH'
        WHEN bowling_team IN ('Rising Pune Supergiants', 'Rising Pune Supergiant') THEN 'RPS'
        WHEN bowling_team = 'Gujarat Lions' THEN 'GL'
        WHEN bowling_team IN ('Royal Challengers Bangalore','Royal Challengers Bengaluru' ) THEN 'RCB'
        WHEN bowling_team = 'Mumbai Indians' THEN 'MI'
        WHEN bowling_team = 'Chennai Super Kings' THEN 'CSK'
        WHEN bowling_team = 'Kolkata Knight Riders' THEN 'KKR'
        WHEN bowling_team = 'Rajasthan Royals' THEN 'RR'
        WHEN bowling_team = 'Lucknow Super Giants' THEN 'LSG'
        WHEN bowling_team = 'Gujarat Titans' THEN 'GT'
        WHEN bowling_team = 'Pune Warriors' THEN 'PWI'
        WHEN bowling_team = 'Kochi Tuskers Kerala' THEN 'KTK'
        ELSE bowling_team
    END AS bowl_team_clean
FROM df_working
""").df()

print("Full Team Standardization Complete ")

Full Team Standardization Complete 


### STANDARDIZE WINNING TEAM
### This ensures 'winning_team' matches our 'bat_team_clean' and 'bowl_team_clean' codes

In [ ]:

df_final = duckdb.query("""
SELECT *,
    CASE
        WHEN winning_team IN ('Kings XI Punjab', 'Punjab Kings') THEN 'PBKS'
        WHEN winning_team IN ('Delhi Daredevils', 'Delhi Capitals') THEN 'DC'
        WHEN winning_team IN ('Deccan Chargers', 'Sunrisers Hyderabad') THEN 'SRH'
        WHEN winning_team IN ('Rising Pune Supergiants', 'Rising Pune Supergiant') THEN 'RPS'
        WHEN winning_team = 'Gujarat Lions' THEN 'GL'
        WHEN winning_team IN ('Royal Challengers Bangalore', 'Royal Challengers Bengaluru') THEN 'RCB'
        WHEN winning_team = 'Mumbai Indians' THEN 'MI'
        WHEN winning_team = 'Chennai Super Kings' THEN 'CSK'
        WHEN winning_team = 'Kolkata Knight Riders' THEN 'KKR'
        WHEN winning_team = 'Rajasthan Royals' THEN 'RR'
        WHEN winning_team = 'Lucknow Super Giants' THEN 'LSG'
        WHEN winning_team = 'Gujarat Titans' THEN 'GT'
        WHEN winning_team = 'Pune Warriors' THEN 'PWI'
        WHEN winning_team = 'Kochi Tuskers Kerala' THEN 'KTK'
        ELSE winning_team
    END AS winning_team_clean
FROM df_final
""").df()

print("✅ Winning Team Standardization Complete!")

✅ Winning Team Standardization Complete!


In [ ]:
df_final = duckdb.query("""
SELECT *,
    CASE
        WHEN toss_winner IN ('Kings XI Punjab', 'Punjab Kings') THEN 'PBKS'
        WHEN toss_winner IN ('Delhi Daredevils', 'Delhi Capitals') THEN 'DC'
        WHEN toss_winner IN ('Deccan Chargers', 'Sunrisers Hyderabad') THEN 'SRH'
        WHEN toss_winner IN ('Rising Pune Supergiants', 'Rising Pune Supergiant') THEN 'RPS'
        WHEN toss_winner = 'Gujarat Lions' THEN 'GL'
        WHEN toss_winner IN ('Royal Challengers Bangalore', 'Royal Challengers Bengaluru') THEN 'RCB'
        WHEN toss_winner = 'Mumbai Indians' THEN 'MI'
        WHEN toss_winner = 'Chennai Super Kings' THEN 'CSK'
        WHEN toss_winner = 'Kolkata Knight Riders' THEN 'KKR'
        WHEN toss_winner = 'Rajasthan Royals' THEN 'RR'
        WHEN toss_winner = 'Lucknow Super Giants' THEN 'LSG'
        WHEN toss_winner= 'Gujarat Titans' THEN 'GT'
        WHEN toss_winner = 'Pune Warriors' THEN 'PWI'
        WHEN toss_winner= 'Kochi Tuskers Kerala' THEN 'KTK'
        ELSE toss_winner
    END AS toss_winner_clean
FROM df_final
""").df()

print("✅ toss winner Team Standardization Complete!")

✅ toss winner Team Standardization Complete!


In [ ]:
df_final.head()

,match_id,season,date,innings,batting_team,bowling_team,over,ball,batter,bowler,...,venue,city,toss_winner,toss_decision,player_of_match,winning_team,win_margin,bat_team_clean,bowl_team_clean,winning_team_clean
0,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,...,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs,KKR,RCB,KKR
1,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,...,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs,KKR,RCB,KKR
2,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,...,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs,KKR,RCB,KKR
3,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,...,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs,KKR,RCB,KKR
4,335982,2007/08,2008-04-18,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,...,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140 runs,KKR,RCB,KKR


In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278205 entries, 0 to 278204
Data columns (total 27 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   match_id            278205 non-null  int64 
 1   season              278205 non-null  object
 2   date                278205 non-null  object
 3   innings             278205 non-null  int64 
 4   batting_team        278205 non-null  object
 5   bowling_team        278205 non-null  object
 6   over                278205 non-null  int64 
 7   ball                278205 non-null  int64 
 8   batter              278205 non-null  object
 9   bowler              278205 non-null  object
 10  runs_batter         278205 non-null  int64 
 11  runs_bowler         278205 non-null  int64 
 12  runs_extras         278205 non-null  int64 
 13  runs_total          278205 non-null  int64 
 14  team_runs           278205 non-null  int64 
 15  wicket_kind         13823 non-null   object
 16  pl

### Check which columns have null values

In [ ]:
print(df_final.isnull().sum())

match_id                   0
season                     0
date                       0
innings                    0
batting_team               0
bowling_team               0
over                       0
ball                       0
batter                     0
bowler                     0
runs_batter                0
runs_bowler                0
runs_extras                0
runs_total                 0
team_runs                  0
wicket_kind           264382
player_out            264382
venue                      0
city                       0
toss_winner                0
toss_decision              0
player_of_match            0
winning_team               0
win_margin              4702
bat_team_clean             0
bowl_team_clean            0
winning_team_clean         0
dtype: int64


## Removing duplicate rows

In [ ]:
# This only removes rows if EVERY single column is an exact match.
duplicate_count = df_final.duplicated().sum()
print(f"Duplicate rows found (with team_runs check): {duplicate_count}")

if duplicate_count > 0:
    df_final = df_final.drop_duplicates()

Duplicate rows found (with team_runs check): 0


## Handle missing values and incorrect data types

In [ ]:
df['season'].unique()

array(['2007/08', '2009', 2009, '2009/10', '2011', 2011, 2012, 2013, 2014,
       2015, 2016, 2017, 2018, 2019, '2019', '2020/21', '2021', 2021,
       2022, 2023, 2024, 2025], dtype=object)

In [ ]:
# 1. Convert to string to ensure consistency
df_final['season'] = df_final['season'].astype(str)

season_map = {
    '2007/08': 2008,  # Inaugural season (Final in 2008)
    '2009/10': 2010,  # 3rd edition (Final in 2010)
    '2020/21': 2020,  # The COVID-delayed season (Played in 2020)
    '2009': 2009,
    '2011': 2011,
    '2019': 2019,
    '2021': 2021
}

# 3. Apply the map, and for anything else, just keep the first 4 digits
def final_season_cleaner(val):
    if val in season_map:
        return season_map[val]
    # For standard years already in 2022, 2023, etc.
    try:
        return int(val[:4])
    except:
        return val

df_final['season'] = df_final['season'].apply(final_season_cleaner)

# 4. Verification
print(f"Unique Seasons: {sorted(df_final['season'].unique())}")
print("\n📊 FIXED MATCH COUNTS (2020 vs 2021):")
print(df_final.groupby('season')['match_id'].nunique())

Unique Seasons: [np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

📊 FIXED MATCH COUNTS (2020 vs 2021):
season
2008    58
2009    57
2010    60
2011    73
2012    74
2013    76
2014    60
2015    59
2016    60
2017    59
2018    60
2019    60
2020    60
2021    60
2022    74
2023    74
2024    71
2025    74
Name: match_id, dtype: int64


## Create a unique list of venues to inspect

In [ ]:
unique_venues = df_final['venue'].unique()
unique_venues.sort()

for v in unique_venues:
    print(v)

Arun Jaitley Stadium
Arun Jaitley Stadium, Delhi
Barabati Stadium
Barsapara Cricket Stadium, Guwahati
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow
Brabourne Stadium
Brabourne Stadium, Mumbai
Buffalo Park
De Beers Diamond Oval
Dr DY Patil Sports Academy
Dr DY Patil Sports Academy, Mumbai
Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium
Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam
Dubai International Cricket Stadium
Eden Gardens
Eden Gardens, Kolkata
Feroz Shah Kotla
Green Park
Himachal Pradesh Cricket Association Stadium
Himachal Pradesh Cricket Association Stadium, Dharamsala
Holkar Cricket Stadium
JSCA International Stadium Complex
Kingsmead
M Chinnaswamy Stadium
M Chinnaswamy Stadium, Bengaluru
M.Chinnaswamy Stadium
MA Chidambaram Stadium
MA Chidambaram Stadium, Chepauk
MA Chidambaram Stadium, Chepauk, Chennai
Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur
Maharaja Yadavindra Singh International Cricket Stadium, Ne

# 1. Define the Master Mapping to fix venues

In [ ]:

venue_mapping = {
    # Existing duplicates and variations
    'Brabourne Stadium': 'Brabourne Stadium, Mumbai',
    'Dr DY Patil Sports Academy': 'Dr DY Patil Sports Academy, Mumbai',
    'Eden Gardens': 'Eden Gardens, Kolkata',
    'Feroz Shah Kotla': 'Arun Jaitley Stadium, Delhi',
    'Arun Jaitley Stadium': 'Arun Jaitley Stadium, Delhi',
    'M Chinnaswamy Stadium': 'M Chinnaswamy Stadium, Bengaluru',
    'M.Chinnaswamy Stadium': 'M Chinnaswamy Stadium, Bengaluru',
    'MA Chidambaram Stadium': 'MA Chidambaram Stadium, Chennai',
    'MA Chidambaram Stadium, Chepauk': 'MA Chidambaram Stadium, Chennai',
    'MA Chidambaram Stadium, Chepauk, Chennai': 'MA Chidambaram Stadium, Chennai',
    'Punjab Cricket Association IS Bindra Stadium': 'IS Bindra Stadium, Mohali',
    'Punjab Cricket Association IS Bindra Stadium, Mohali': 'IS Bindra Stadium, Mohali',
    'Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh': 'IS Bindra Stadium, Mohali',
    'Punjab Cricket Association Stadium, Mohali': 'IS Bindra Stadium, Mohali',
    'Rajiv Gandhi International Stadium': 'Rajiv Gandhi International Stadium, Hyderabad',
    'Rajiv Gandhi International Stadium, Uppal': 'Rajiv Gandhi International Stadium, Hyderabad',
    'Rajiv Gandhi International Stadium, Uppal, Hyderabad': 'Rajiv Gandhi International Stadium, Hyderabad',
    'Wankhede Stadium': 'Wankhede Stadium, Mumbai',
    'Sardar Patel Stadium, Motera': 'Narendra Modi Stadium, Ahmedabad',
    'Sawai Mansingh Stadium': 'Sawai Mansingh Stadium, Jaipur',
    'Maharashtra Cricket Association Stadium': 'MCA Stadium, Pune',
    'Subrata Roy Sahara Stadium': 'MCA Stadium, Pune',
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow': 'Ekana Cricket Stadium, Lucknow',
    'Himachal Pradesh Cricket Association Stadium': 'HPCA Stadium, Dharamsala',
    'Himachal Pradesh Cricket Association Stadium, Dharamsala': 'HPCA Stadium, Dharamsala',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'ACA-VDCA Stadium, Visakhapatnam',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam': 'ACA-VDCA Stadium, Visakhapatnam',
    'JSCA International Stadium Complex': 'JSCA Stadium, Ranchi',
    'Saurashtra Cricket Association Stadium': 'SCA Stadium, Rajkot',
    'Vidarbha Cricket Association Stadium, Jamtha': 'VCA Stadium, Nagpur',
    'Shaheed Veer Narayan Singh International Stadium': 'Raipur International Stadium, Raipur',
    'Maharaja Yadavindra Singh International Cricket Stadium, Mullanpur': 'MYS International Stadium, Mullanpur',
    'Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh': 'MYS International Stadium, Mullanpur',

    # Overseas Venues
    'Dubai International Cricket Stadium': 'Dubai International Stadium, UAE',
    'Zayed Cricket Stadium, Abu Dhabi': 'Sheikh Zayed Stadium, Abu Dhabi',
    'Sheikh Zayed Stadium': 'Sheikh Zayed Stadium, Abu Dhabi',
    'Sharjah Cricket Stadium': 'Sharjah Cricket Stadium, UAE',
    'Newlands': 'Newlands, Cape Town',
    'Kingsmead': 'Kingsmead, Durban',
    'St George\'s Park': 'St George\'s Park, Gqeberha',
    'SuperSport Park': 'SuperSport Park, Centurion',
    'New Wanderers Stadium': 'Wanderers Stadium, Johannesburg',
    'Buffalo Park': 'Buffalo Park, East London',
    'De Beers Diamond Oval': 'De Beers Diamond Oval, Kimberley',
    'OUTsurance Oval': 'Mangaung Oval, Bloemfontein'
}

# 2. Apply the mapping
df_final['venue_clean'] = df_final['venue'].replace(venue_mapping)

# 3. Handle any missed cases with a quick string strip and title case
df_final['venue_clean'] = df_final['venue_clean'].str.strip()

# 4. Final verification print
unique_list = sorted(df_final['venue_clean'].unique())
print(f"Total Unique Venues: {len(unique_list)}")
print("-" * 30)
for venue in unique_list:
    print(venue)

Total Unique Venues: 37
------------------------------
ACA-VDCA Stadium, Visakhapatnam
Arun Jaitley Stadium, Delhi
Barabati Stadium
Barsapara Cricket Stadium, Guwahati
Brabourne Stadium, Mumbai
Buffalo Park, East London
De Beers Diamond Oval, Kimberley
Dr DY Patil Sports Academy, Mumbai
Dubai International Stadium, UAE
Eden Gardens, Kolkata
Ekana Cricket Stadium, Lucknow
Green Park
HPCA Stadium, Dharamsala
Holkar Cricket Stadium
IS Bindra Stadium, Mohali
JSCA Stadium, Ranchi
Kingsmead, Durban
M Chinnaswamy Stadium, Bengaluru
MA Chidambaram Stadium, Chennai
MCA Stadium, Pune
MYS International Stadium, Mullanpur
Maharashtra Cricket Association Stadium, Pune
Mangaung Oval, Bloemfontein
Narendra Modi Stadium, Ahmedabad
Nehru Stadium
Newlands, Cape Town
Raipur International Stadium, Raipur
Rajiv Gandhi International Stadium, Hyderabad
SCA Stadium, Rajkot
Sawai Mansingh Stadium, Jaipur
Sharjah Cricket Stadium, UAE
Sheikh Zayed Stadium, Abu Dhabi
St George's Park, Gqeberha
SuperSport Park, Ce

## --- FINAL MASTER CLEANING PIPELINE ---

In [ ]:

# 1. Row-Level Deduplication (Preserving Extras)
df_final = df_final.drop_duplicates()

# 2. Final Column Selection & Renaming for Analysis
# This ensures we use our 'clean' versions of teams and venues
df_final = df_final[[
    'match_id', 'season', 'date', 'innings', 'bat_team_clean', 'bowl_team_clean',
    'over', 'ball', 'batter', 'bowler', 'runs_batter','runs_bowler', 'runs_extras', 'runs_total','team_runs',
    'wicket_kind', 'player_out', 'venue_clean', 'city', 'toss_winner_clean', 'toss_decision',
    'player_of_match', 'winning_team_clean', 'win_margin'
]]

# 3. Final Integrity Audit
print("📊 FINAL DATA AUDIT")
print("-" * 30)
print(f"Total Balls: {len(df_final)}")
print(f"Total Matches: {df_final['match_id'].nunique()}")
print(f"Seasons Covered: {df_final['season'].min()} to {df_final['season'].max()}")
print(f"Unique Teams: {df_final['bat_team_clean'].nunique()}")
print(f"Unique Venues: {df_final['venue_clean'].nunique()}")

# 4. Exporting the 'Gold' Dataset
df_final.to_csv('/content/drive/MyDrive/IPL_PROJECT/02_data_cleaned/IPL_Cleaned_Final.csv', index=False)
print("-" * 30)
print(" SUCCESS: Cleaned file saved to 02_data_cleaned/IPL_Cleaned_Final.csv")

📊 FINAL DATA AUDIT
------------------------------
Total Balls: 278205
Total Matches: 1169
Seasons Covered: 2008 to 2025
Unique Teams: 14
Unique Venues: 37
------------------------------
📂 SUCCESS: Cleaned file saved to 02_data_cleaned/IPL_Cleaned_Final.csv


# **Key Observations:**

### Dataset contained missing values in specific columns which were handled appropriately
### Data types were standardized for consistency
### Column names were cleaned for easier reference in analysis